# Build a Diffusion Model: 2D Spiral

In this notebook we build a **Denoising Diffusion Probabilistic Model (DDPM)** from scratch and train it to generate 2D spiral data.

Everything runs on CPU in under a minute — no GPU needed.

This notebook mirrors the diffusion lesson on the [mlearn website](https://mlearn.org). Head there for the full explanation of the math and intuition behind each step.

---

**How to use this notebook:** Each section has an **exercise cell** where you write code yourself, followed by a **collapsed solution cell** you can expand if you get stuck. Try the exercise first!

## 1 — Setup

### Python for JS/TS Devs — Quick Reference

A few syntax differences you'll see throughout this notebook:

- **`f"text {var}"`** — like JS template literals `` `text ${var}` ``
- **`x: int = 5`** — type hints, like TypeScript's `x: number = 5`
- **`def fn() -> int:`** — return type annotation, like `: number` in TS
- **`None`** — Python's `null` (there is no `undefined`)
- **`lambda x: x + 1`** — like `(x) => x + 1`
- **`for i in range(5):`** — like `for (let i = 0; i < 5; i++)`
- **`a, b = (1, 2)`** — tuple unpacking, like `const [a, b] = [1, 2]`
- **`x[1:3]`** — slicing, like `x.slice(1, 3)`
- **`import X as Y`** — like `import * as Y from 'X'`

PyTorch-specific things are explained inline as they come up.

In [ ]:
!pip install torch matplotlib -q

import torch              # main ML framework (like TensorFlow but more Pythonic)
import torch.nn as nn     # neural network module — "as nn" is like: import * as nn from 'torch.nn'
import numpy as np        # numeric arrays — predates torch, still used for constants like np.pi
import matplotlib.pyplot as plt  # plotting library
from typing import Optional      # like TypeScript's `| undefined`

torch.manual_seed(42)     # fixed RNG seed for reproducibility
np.random.seed(42)

# f-string: like JS template literal `PyTorch ${torch.__version__}`
print(f"PyTorch {torch.__version__}")

## 2 — The Spiral Dataset

We generate points that lie on a 2D spiral. This is the distribution our diffusion model will learn to sample from.

### Exercise 1: Generate the Spiral

Write a function `make_spiral` that generates `n_points` 2D points lying on a spiral.

**Recipe:**
1. Create a parameter `t` that goes from `0` to `4π` (use `torch.linspace`).
2. Compute a radius `r = t / (4π)` so the spiral expands outward.
3. Convert to Cartesian: `x = r * cos(t)`, `y = r * sin(t)`, plus a little Gaussian noise.
4. Stack into a `(n_points, 2)` tensor.

Then create the dataset and plot it with `plt.scatter`.

In [ ]:
def make_spiral(n_points: int = 2000, noise: float = 0.3) -> torch.Tensor:
    """Generate 2D spiral dataset."""
    # Step 1: Create parameter t from 0 to 4*pi
    # torch.linspace(a, b, n) — creates n evenly spaced values from a to b
    # like: Array.from({length: n}, (_, i) => a + i * (b - a) / (n - 1))
    # YOUR CODE HERE

    # Step 2: Radius grows linearly with t
    # YOUR CODE HERE

    # Step 3: Convert to (x, y) with some noise
    # torch.randn(n) — n random samples from normal distribution (mean=0, std=1)
    # Hint: x = r * cos(t) + noise * randn * 0.1
    # YOUR CODE HERE

    # Step 4: Stack into (n_points, 2) tensor
    # torch.stack([a, b], dim=1) — like zipping two arrays into pairs: [[a[0],b[0]], [a[1],b[1]], ...]
    # YOUR CODE HERE
    pass  # None return — like returning undefined in JS

dataset = make_spiral()

# Plot the spiral
plt.figure(figsize=(5, 5))
# YOUR CODE HERE: scatter plot of dataset
# dataset[:, 0] — slicing: all rows, column 0. Like dataset.map(row => row[0])
plt.title("Spiral Dataset")
plt.axis("equal")
plt.show()

In [ ]:
#@title Solution (click to expand)
def make_spiral(n_points: int = 2000, noise: float = 0.3) -> torch.Tensor:
    """Generate 2D spiral dataset."""
    # torch.linspace(start, end, steps) — evenly spaced values, like a range generator
    t = torch.linspace(0, 4 * np.pi, n_points)
    r = t / (4 * np.pi)  # radius grows linearly — element-wise division (like .map(v => v / max))
    # torch.cos/sin operate element-wise on entire tensors (no .map() needed)
    # torch.randn(n) — n samples from standard normal distribution (bell curve, mean=0)
    x = r * torch.cos(t) + noise * torch.randn(n_points) * 0.1
    y = r * torch.sin(t) + noise * torch.randn(n_points) * 0.1
    # torch.stack — combines tensors along a new dimension
    # dim=1 means stack as columns → shape (n_points, 2), like: points.map((_, i) => [x[i], y[i]])
    data = torch.stack([x, y], dim=1)  # (n_points, 2)
    return data

dataset = make_spiral()

plt.figure(figsize=(5, 5))
# dataset[:, 0] slices all rows, column 0 — like dataset.map(row => row[0])
# s=3 is dot size, alpha=0.5 is opacity
plt.scatter(dataset[:, 0], dataset[:, 1], s=3, alpha=0.5)
plt.title("Spiral Dataset")
plt.axis("equal")
plt.show()

## 3 — Forward Process (Adding Noise)

The forward process gradually destroys structure by adding Gaussian noise according to a variance schedule.

Given clean data $x_0$, we can jump to any timestep $t$ in closed form:

$$q(x_t | x_0) = \mathcal{N}(x_t;\; \sqrt{\bar\alpha_t}\, x_0,\; (1 - \bar\alpha_t)\, I)$$

### Exercise 2: Noise Schedule and Forward Diffusion

Build the noise schedule and the forward diffusion function.

**Step A — Noise schedule:**
- `betas`: linearly spaced from `1e-4` to `0.02`, length `T=300`
- `alphas = 1 - betas`
- `alpha_bars`: cumulative product of `alphas` (use `torch.cumprod`)

**Step B — `forward_diffusion(x_0, t, noise)`:**
- If `noise` is None, sample from standard normal
- Look up `alpha_bars[t]` and reshape to `(batch, 1)` for broadcasting
- Compute: `x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * noise`
- Return `(x_t, noise)`

In [ ]:
# Noise schedule
T = 300  # total diffusion timesteps

# Step A: Build the schedule
# torch.linspace — same as before, evenly spaced floats
# betas = ...           # linear schedule from 1e-4 to 0.02
# alphas = ...          # 1 - betas (element-wise, like betas.map(b => 1 - b))
# torch.cumprod(x, dim=0) — cumulative product along an axis
#   like: x.reduce((acc, v, i) => [...acc, acc[i-1] * v], [x[0]])
# alpha_bars = ...      # cumulative product of alphas
# YOUR CODE HERE


def forward_diffusion(
    x_0: torch.Tensor,
    t: torch.Tensor,
    noise: Optional[torch.Tensor] = None,   # Optional[X] is like X | undefined in TS
) -> tuple[torch.Tensor, torch.Tensor]:      # -> return type annotation, like (): [Tensor, Tensor] in TS
    """
    Apply forward diffusion to x_0 at timesteps t.
    Returns (x_t, noise).
    """
    # Step B: Implement the closed-form forward diffusion
    # torch.randn_like(x) — random normal tensor same shape as x
    # alpha_bars[t] — fancy indexing: grabs elements at indices in t (like t.map(i => alpha_bars[i]))
    # .unsqueeze(-1) — adds a dimension at the end, shape (N,) → (N, 1), needed for broadcasting
    # Hint: ab = alpha_bars[t].unsqueeze(-1) for shape (batch, 1)
    # YOUR CODE HERE
    pass

In [ ]:
#@title Solution (click to expand)
# Noise schedule
T = 300  # total diffusion timesteps

betas = torch.linspace(1e-4, 0.02, T)          # linear schedule — 300 evenly spaced values
alphas = 1.0 - betas                             # element-wise subtraction (like .map(b => 1 - b))
# torch.cumprod — cumulative product: [a, a*b, a*b*c, ...]
# like: alphas.reduce((acc, v) => [...acc, acc.at(-1) * v], [alphas[0]])
alpha_bars = torch.cumprod(alphas, dim=0)

def forward_diffusion(
    x_0: torch.Tensor,
    t: torch.Tensor,
    noise: Optional[torch.Tensor] = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Apply forward diffusion to x_0 at timesteps t.
    Returns (x_t, noise).
    """
    if noise is None:                    # None is Python's null
        noise = torch.randn_like(x_0)   # random normal, same shape as x_0
    # alpha_bars[t] — tensor indexing with another tensor: grabs one value per batch element
    # .unsqueeze(-1) — reshapes (batch,) → (batch, 1) so it broadcasts with (batch, 2)
    ab = alpha_bars[t].unsqueeze(-1)  # (batch, 1)
    # torch.sqrt — element-wise square root (like Math.sqrt but on whole tensors)
    x_t = torch.sqrt(ab) * x_0 + torch.sqrt(1.0 - ab) * noise
    return x_t, noise  # returns a tuple — like returning [x_t, noise] in JS

### Visualize the Forward Process

Run this cell to see the spiral progressively destroyed by noise.

In [ ]:
# Visualize the forward process at several timesteps
fig, axes = plt.subplots(1, 5, figsize=(20, 4))  # creates 5 subplots; tuple unpacking like const [fig, axes] = ...
steps_to_show = [0, 50, 100, 200, 299]           # Python list — same as JS array

# zip(axes, steps_to_show) — pairs up elements from both lists
# like lodash _.zip(axes, steps_to_show), then iterated with destructuring
for ax, step in zip(axes, steps_to_show):
    # torch.full((n,), value) — creates tensor of n copies of value, like Array(n).fill(value)
    t_vec = torch.full((dataset.shape[0],), step, dtype=torch.long)  # dtype=torch.long means integer (int64)
    x_t, _ = forward_diffusion(dataset, t_vec)  # _ is convention for "I don't need this value"
    # .numpy() converts a torch tensor to a numpy array (needed for matplotlib)
    ax.scatter(x_t[:, 0].numpy(), x_t[:, 1].numpy(), s=2, alpha=0.4)
    ax.set_title(f"t = {step}")  # f-string: like `t = ${step}`
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")

plt.suptitle("Forward Diffusion: Spiral \u2192 Noise", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 4 — The Noise Predictor

We train a small MLP that takes a noisy point $x_t$ and the timestep $t$, and predicts the noise $\epsilon$ that was added.

The timestep is encoded with **sinusoidal positional embeddings** (same idea as in Transformers).

> **Python for JS devs — `nn.Module`:** PyTorch neural networks are classes that extend `nn.Module` (like `class MyNet extends nn.Module`). You define layers in `__init__` (the constructor) and implement `forward()` which is the method called when you do `model(input)` — think of it as overriding a `call()` method. `nn.Sequential` is like chaining layers in a pipeline — data flows through each one in order.

### Exercise 3: Sinusoidal Embedding and Noise Predictor Network

This is one of the trickiest parts. You need to build two things:

**A) `sinusoidal_embedding(t, dim)`** — Maps integer timesteps to a vector.
- Compute `half = dim // 2` frequency bands
- `freqs = exp(-log(10000) * arange(half) / half)`
- Multiply: `args = t_float[:, None] * freqs[None, :]` giving shape `(batch, half)`
- Concatenate `sin(args)` and `cos(args)` along the last dimension → `(batch, dim)`

**B) `NoisePredictor(nn.Module)`** — An MLP that predicts 2D noise.
- `__init__`: Create an `nn.Sequential` with:
  - `Linear(2 + time_dim, hidden_dim)` → `ReLU` → `Linear(hidden_dim, hidden_dim)` → `ReLU` → `Linear(hidden_dim, 2)`
- `forward(x_t, t)`:
  1. Compute time embedding: `t_emb = sinusoidal_embedding(t, self.time_dim)` → `(batch, time_dim)`
  2. Concatenate `[x_t, t_emb]` along dim=-1 → `(batch, 2 + time_dim)`
  3. Pass through `self.net`

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int = 64) -> torch.Tensor:
    """
    Map integer timesteps to sinusoidal embeddings of size `dim`.
    t: (batch,) long tensor
    returns: (batch, dim)
    """
    half = dim // 2  # // is integer division (floors result), like Math.floor(dim / 2)

    # Compute frequency bands
    # torch.arange(n) — like Array.from({length: n}, (_, i) => i)
    # torch.exp — element-wise e^x (like Math.exp but on entire tensor)
    # freqs = exp(-log(10000) * arange(half) / half)
    # YOUR CODE HERE

    # t.float() — cast int tensor to float (like Number(t))
    # .unsqueeze(-1) — add trailing dim: (batch,) → (batch, 1) for broadcasting
    # * between tensors is element-wise multiply (broadcasting fills in dimensions)
    # Compute args = t_float[:, None] * freqs[None, :]
    # YOUR CODE HERE

    # torch.cat([a, b], dim=-1) — concatenate along last axis
    # like [...arrayA, ...arrayB] but along a specific dimension
    # Return concatenation of sin(args) and cos(args)
    # YOUR CODE HERE
    pass


class NoisePredictor(nn.Module):
    # nn.Module — base class for all neural networks (like extending a base Component class)
    def __init__(self, hidden_dim: int = 128, time_dim: int = 64):
        super().__init__()  # must call parent constructor, like super() in JS classes
        self.time_dim = time_dim  # self.x is like this.x in JS

        # nn.Sequential — chains layers: output of each feeds into the next
        # nn.Linear(in, out) — a dense/fully-connected layer: y = Wx + b
        # nn.ReLU() — activation function: max(0, x)
        # Build the MLP: input is [x_t (2 dims), time_emb (time_dim dims)]
        self.net = nn.Sequential(
            # YOUR CODE HERE
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # forward() is called when you do model(x_t, t) — like a __call__ override
        # Step 1: Get time embedding
        # YOUR CODE HERE

        # Step 2: torch.cat([a, b], dim=-1) — concatenate along last dimension
        # YOUR CODE HERE

        # Step 3: Pass through self.net and return
        # YOUR CODE HERE
        pass


model = NoisePredictor()
# Generator expression: sum(p.numel() for p in ...) — like [...params].reduce((s, p) => s + p.numel(), 0)
# p.numel() returns number of elements in tensor
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title Solution (click to expand)
def sinusoidal_embedding(t: torch.Tensor, dim: int = 64) -> torch.Tensor:
    """
    Map integer timesteps to sinusoidal embeddings of size `dim`.
    t: (batch,) long tensor
    returns: (batch, dim)
    """
    half = dim // 2  # // is integer division (floors result), like Math.floor(dim / 2)
    # torch.arange(half) — [0, 1, 2, ..., half-1], like Array.from({length: half}, (_, i) => i)
    freqs = torch.exp(
        -np.log(10000.0) * torch.arange(half, dtype=torch.float32) / half
    )
    # t.float() — cast to float type (like Number(t))
    # .unsqueeze(-1) — (batch,) → (batch, 1); [None, :] adds dim at front: (half,) → (1, half)
    # broadcasting: (batch, 1) * (1, half) → (batch, half) — each t scaled by each freq
    args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)  # (batch, half)
    # torch.cat — concatenate tensors, like [...sin, ...cos] but along a tensor dimension
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (batch, dim)


class NoisePredictor(nn.Module):
    # nn.Module — base class for all neural networks (like extending a base class)
    # __init__ is the constructor (like constructor() in JS)
    def __init__(self, hidden_dim: int = 128, time_dim: int = 64):
        super().__init__()  # must call parent constructor, like super() in JS
        self.time_dim = time_dim  # self.x is like this.x in JS
        # nn.Sequential — chains layers into a pipeline (data flows through in order)
        # nn.Linear(in, out) — fully-connected layer: y = Wx + b
        # nn.ReLU() — activation function: max(0, x), adds non-linearity
        self.net = nn.Sequential(
            nn.Linear(2 + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),  # predict 2D noise
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # forward() is called when you do model(x_t, t) — like overriding a __call__ method
        t_emb = sinusoidal_embedding(t, self.time_dim)  # (batch, time_dim)
        # torch.cat([a, b], dim=-1) — concat along last dim, like [...a, ...b] per row
        inp = torch.cat([x_t, t_emb], dim=-1)           # (batch, 2 + time_dim)
        return self.net(inp)                              # (batch, 2) — calling self.net runs forward()


model = NoisePredictor()
# sum(... for p in ...) — generator expression, like [...params].reduce((s, p) => s + p.numel(), 0)
# p.numel() — number of elements in a tensor (like .length for a flat array)
# :, in f-string formats number with commas: 1000 → "1,000"
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5 — Training

Each training step:
1. Sample a batch of clean points $x_0$ from the dataset.
2. Sample random timesteps $t \sim \text{Uniform}\{0, \ldots, T{-}1\}$.
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$.
4. Compute the noisy version $x_t$ via the forward process.
5. Predict $\hat\epsilon = f_\theta(x_t, t)$.
6. Minimize $\|\epsilon - \hat\epsilon\|^2$.

### Exercise 4: Training Loop

Implement the training loop following the 6 steps above.

**Settings:** `Adam` optimizer with `lr=1e-3`, `batch_size=256`, `n_steps=5000`.

**Inside each step:**
1. Sample random indices into `dataset` and grab `x_0`
2. Sample random timesteps `t` in `[0, T)`
3. Sample noise from `torch.randn_like(x_0)`
4. Compute `x_t` via `forward_diffusion`
5. Get `noise_pred = model(x_t, t)`
6. Compute MSE loss between `noise_pred` and `noise`, backprop, step

Print loss every 1000 steps and plot the loss curve at the end.

In [ ]:
# torch.optim.Adam — the Adam optimizer (adjusts model weights to minimize loss)
# model.parameters() — returns all trainable weights (like getting all the knobs to tune)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 256
n_steps = 5000
losses = []  # plain Python list — same as JS array

# range(n_steps) — generates 0, 1, 2, ..., n_steps-1 (like for (let step = 0; step < n_steps; step++))
for step in range(n_steps):
    # 1. Sample batch of clean points
    # torch.randint(low, high, (size,)) — random ints in [low, high), like Math.floor(Math.random() * (high - low)) + low
    # YOUR CODE HERE

    # 2. Sample random timesteps
    # YOUR CODE HERE

    # 3. Sample noise
    # YOUR CODE HERE

    # 4. Forward diffusion to get x_t
    # YOUR CODE HERE

    # 5. Predict noise with model
    # model(x_t, t) calls model.forward(x_t, t) under the hood
    # YOUR CODE HERE

    # 6. MSE loss, backward, optimizer step
    # nn.functional.mse_loss — mean squared error between prediction and target
    # optimizer.zero_grad() — clear old gradients (PyTorch accumulates them by default)
    # loss.backward() — compute gradients via automatic differentiation (backpropagation)
    # optimizer.step() — update weights using the computed gradients
    # YOUR CODE HERE

    # loss.item() — extracts a plain Python float from a single-element tensor
    # like unwrapping a boxed number to a primitive
    losses.append(loss.item())
    if (step + 1) % 1000 == 0:  # % is modulo, same as JS
        print(f"Step {step+1:5d} | Loss: {loss.item():.4f}")  # :5d = pad to 5 digits, :.4f = 4 decimal places

plt.figure(figsize=(8, 3))
plt.plot(losses, linewidth=0.5)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
#@title Solution (click to expand)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam optimizer with learning rate 1e-3
batch_size = 256
n_steps = 5000
losses = []

for step in range(n_steps):  # range(n) — 0 to n-1, like for (let i = 0; i < n; i++)
    # 1. Sample batch
    idx = torch.randint(0, len(dataset), (batch_size,))  # len() is like .length
    x_0 = dataset[idx]  # fancy indexing: pick rows by index array (like idx.map(i => dataset[i]))

    # 2. Sample random timesteps
    t = torch.randint(0, T, (batch_size,))  # random ints in [0, T)

    # 3. Sample noise
    noise = torch.randn_like(x_0)  # same shape as x_0, filled with random normal values

    # 4. Forward diffusion
    x_t, _ = forward_diffusion(x_0, t, noise)  # _ means "ignore this return value"

    # 5. Predict noise
    noise_pred = model(x_t, t)  # calls model.forward(x_t, t) — Python magic method

    # 6. MSE loss
    loss = nn.functional.mse_loss(noise_pred, noise)  # mean squared error

    optimizer.zero_grad()  # clear gradients from previous step (PyTorch accumulates by default)
    loss.backward()        # autograd: compute d(loss)/d(each weight) via chain rule (backpropagation)
    optimizer.step()       # update weights: weight -= lr * gradient

    losses.append(loss.item())  # .item() extracts plain float from single-element tensor
    if (step + 1) % 1000 == 0:
        print(f"Step {step+1:5d} | Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, linewidth=0.5)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()

## 6 — Sampling (Reverse Process)

Starting from pure noise $x_T \sim \mathcal{N}(0, I)$, we iteratively denoise using the DDPM update rule:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\,\hat\epsilon_\theta(x_t, t)\right) + \sigma_t\, z$$

where $z \sim \mathcal{N}(0, I)$ for $t > 1$ and $z = 0$ for $t = 1$.

> **Python for JS devs — `@torch.no_grad()` and `model.eval()`:** The `@decorator` syntax is like wrapping a function — `@torch.no_grad()` on a function means "don't track gradients inside here" (saves memory during inference, since we only need gradients during training). `reversed(range(T))` iterates backwards — like `for (let i = T-1; i >= 0; i--)`.

### Exercise 5: DDPM Sampling Loop

This is the other tricky part. Implement the reverse sampling loop.

**Function signature:** `ddpm_sample(model, n_samples=2000, record_steps=None)`

**Algorithm:**
1. Start from pure noise: `x = torch.randn(n_samples, 2)`
2. Loop `t_idx` from `T-1` down to `0`:
   - Create a batch of timesteps: `t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)`
   - Predict noise: `eps_pred = model(x, t_batch)`
   - Look up `beta_t`, `alpha_t`, `alpha_bar_t` for this timestep
   - Compute the mean: `mean = (1/sqrt(alpha_t)) * (x - (beta_t / sqrt(1 - alpha_bar_t)) * eps_pred)`
   - If `t_idx > 0`: add noise `x = mean + sqrt(beta_t) * randn_like(x)`
   - If `t_idx == 0`: `x = mean` (no noise at the last step)
   - Optionally record snapshots
3. Return `(x, snapshots)`

**Important:** Wrap the function with `@torch.no_grad()` since we are doing inference.

In [ ]:
# @torch.no_grad() — decorator: disables gradient tracking inside this function
# saves memory during inference (we only need gradients when training)
# like wrapping the function: const ddpm_sample = torch.no_grad(() => { ... })
@torch.no_grad()
def ddpm_sample(model, n_samples: int = 2000, record_steps: list = None):
    """
    DDPM reverse sampling.
    Returns final samples and (optionally) snapshots at recorded steps.
    """
    # Step 1: Start from pure noise
    x = torch.randn(n_samples, 2)  # random normal tensor, shape (n_samples, 2)
    snapshots = {}                  # dict — like a JS object/Map: { key: value }

    # reversed(range(T)) — iterates T-1, T-2, ..., 1, 0
    # like: for (let t_idx = T-1; t_idx >= 0; t_idx--)
    for t_idx in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)

        # Predict noise
        # eps_pred = ...
        # YOUR CODE HERE

        # Look up schedule values for this timestep
        # betas[t_idx] — simple indexing, like betas[t_idx] in JS
        # YOUR CODE HERE

        # Compute the denoised mean
        # YOUR CODE HERE

        # Add noise if t_idx > 0, otherwise x = mean
        # torch.randn_like(x) — random normal, same shape as x
        # YOUR CODE HERE

        # Record snapshot if requested
        if record_steps is not None and t_idx in record_steps:  # "in" checks membership, like .includes()
            snapshots[t_idx] = x.clone()  # .clone() — deep copy, like structuredClone()

    return x, snapshots  # tuple return — like returning [x, snapshots]


# Test: sample and visualize
record_at = [299, 200, 100, 50, 0]
# Tuple unpacking: like const [samples, snaps] = ddpm_sample(...)
samples, snaps = ddpm_sample(model, n_samples=2000, record_steps=record_at)

fig, axes = plt.subplots(1, len(record_at), figsize=(20, 4))  # len() is like .length
for ax, t_idx in zip(axes, record_at):  # zip pairs elements: like _.zip(axes, record_at)
    pts = snaps[t_idx].numpy()  # dict access — like snaps[t_idx] in JS (or snaps.get(t_idx))
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4)
    ax.set_title(f"t = {t_idx}")
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect("equal")

plt.suptitle("Reverse Process: Noise \u2192 Spiral", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#@title Solution (click to expand)
@torch.no_grad()  # decorator — disables autograd (no gradient tracking = faster + less memory)
def ddpm_sample(model, n_samples: int = 2000, record_steps: list = None):
    """
    DDPM reverse sampling.
    Returns final samples and (optionally) snapshots at recorded steps.
    """
    x = torch.randn(n_samples, 2)  # start from pure noise
    snapshots = {}  # Python dict — like a JS object { }

    # reversed(range(T)) — counts down: T-1, T-2, ..., 0
    for t_idx in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)

        # Predict noise — model(x, t) calls model.forward(x, t)
        eps_pred = model(x, t_batch)

        # DDPM update — look up schedule values by index
        beta_t = betas[t_idx]
        alpha_t = alphas[t_idx]
        alpha_bar_t = alpha_bars[t_idx]

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
        )

        if t_idx > 0:
            sigma = torch.sqrt(beta_t)
            x = mean + sigma * torch.randn_like(x)  # add stochastic noise
        else:
            x = mean  # last step: no noise added

        if record_steps is not None and t_idx in record_steps:  # "in" is like .includes()
            snapshots[t_idx] = x.clone()  # .clone() — deep copy (like structuredClone)

    return x, snapshots


# Sample and visualize intermediate steps
record_at = [299, 200, 100, 50, 0]
samples, snaps = ddpm_sample(model, n_samples=2000, record_steps=record_at)

fig, axes = plt.subplots(1, len(record_at), figsize=(20, 4))
for ax, t_idx in zip(axes, record_at):  # zip — pairs up elements from two iterables
    pts = snaps[t_idx].numpy()  # .numpy() converts tensor → numpy array for plotting
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4)
    ax.set_title(f"t = {t_idx}")
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect("equal")

plt.suptitle("Reverse Process: Noise \u2192 Spiral", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Compare Real vs Generated

Run this cell to see the real data side-by-side with the generated samples.

In [ ]:
# Compare generated samples to real data
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# .numpy() — converts tensor to numpy array (matplotlib needs numpy, not torch)
axes[0].scatter(dataset[:, 0].numpy(), dataset[:, 1].numpy(), s=2, alpha=0.4)
axes[0].set_title("Real Data")
axes[0].set_xlim(-2, 2)
axes[0].set_ylim(-2, 2)
axes[0].set_aspect("equal")

axes[1].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=2, alpha=0.4, color="tab:orange")
axes[1].set_title("Generated Samples")
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect("equal")

plt.suptitle("Real vs Generated", fontsize=14)
plt.tight_layout()
plt.show()

## 7 — Experiments: Other 2D Datasets

The same model architecture and training loop work on other shapes. Let's try **moons**, **circles**, and a **figure-8**.

### Exercise 6: Alternative Dataset Generators

Write three dataset generators. Each returns a `(n_points, 2)` tensor.

**A) `make_moons`** — Two interleaving half-circles:
- `theta` from `0` to `pi`, `n = n_points // 2`
- Top moon: `(cos(theta), sin(theta))`
- Bottom moon: `(1 - cos(theta), 1 - sin(theta) - 0.5)`
- Concatenate, add noise, center by subtracting the mean

**B) `make_circles`** — Two concentric circles:
- `theta` from `0` to `2*pi`, `n = n_points // 2`
- Outer: `(cos(theta), sin(theta))`
- Inner: `0.5 * (cos(theta), sin(theta))`
- Concatenate, add noise

**C) `make_figure8`** — Lemniscate of Bernoulli:
- `t` from `0` to `2*pi`
- `x = cos(t) / (1 + sin(t)^2)`
- `y = sin(t) * cos(t) / (1 + sin(t)^2)`
- Add noise

In [ ]:
def make_moons(n_points: int = 2000, noise: float = 0.05) -> torch.Tensor:
    """Two interleaving half-circles."""
    n = n_points // 2  # // is integer division (like Math.floor(n_points / 2))
    # YOUR CODE HERE
    pass


def make_circles(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Two concentric circles."""
    n = n_points // 2
    # YOUR CODE HERE
    pass


def make_figure8(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Lemniscate of Bernoulli (figure-8)."""
    # ** is exponentiation (like Math.pow), so sin(t) ** 2 is sin(t)^2
    # YOUR CODE HERE
    pass


# Preview all datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# zip with tuple unpacking — each iteration destructures both the axis and the (name, gen) pair
# like: for (const [ax, [name, gen]] of _.zip(axes, datasets))
for ax, (name, gen) in zip(axes, [("Moons", make_moons), ("Circles", make_circles), ("Figure-8", make_figure8)]):
    d = gen()  # call the function — gen is a variable holding a function reference (like a callback)
    ax.scatter(d[:, 0].numpy(), d[:, 1].numpy(), s=2, alpha=0.5)
    ax.set_title(name)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
#@title Solution (click to expand)
def make_moons(n_points: int = 2000, noise: float = 0.05) -> torch.Tensor:
    """Two interleaving half-circles."""
    n = n_points // 2  # // integer division — like Math.floor(n_points / 2)
    theta = torch.linspace(0, np.pi, n)
    # torch.stack([a, b], dim=1) — zip two 1D tensors into (n, 2) matrix
    x_top = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    x_bot = torch.stack([1.0 - torch.cos(theta), 1.0 - torch.sin(theta) - 0.5], dim=1)
    # torch.cat — concatenate along existing dimension (vs stack which adds a new one)
    # like [...x_top, ...x_bot] (appending rows)
    data = torch.cat([x_top, x_bot], dim=0)
    data += noise * torch.randn_like(data)  # += is in-place add, same as JS
    # .mean(dim=0) — average across rows, returns one value per column
    data -= data.mean(dim=0)  # center the data
    return data


def make_circles(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Two concentric circles."""
    n = n_points // 2
    theta = torch.linspace(0, 2 * np.pi, n)
    outer = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    inner = 0.5 * torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)  # scalar * tensor scales all elements
    data = torch.cat([outer, inner], dim=0)
    data += noise * torch.randn_like(data)
    return data


def make_figure8(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Lemniscate of Bernoulli (figure-8)."""
    t = torch.linspace(0, 2 * np.pi, n_points)
    scale = 1.0
    # ** is exponentiation — like Math.pow(). So sin(t) ** 2 means sin(t)^2
    x = scale * torch.cos(t) / (1 + torch.sin(t) ** 2)
    y = scale * torch.sin(t) * torch.cos(t) / (1 + torch.sin(t) ** 2)
    data = torch.stack([x, y], dim=1)
    data += noise * torch.randn_like(data)
    return data


# Preview all datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Nested unpacking: (name, gen) destructures the tuple, like const [name, gen] = pair
for ax, (name, gen) in zip(axes, [("Moons", make_moons), ("Circles", make_circles), ("Figure-8", make_figure8)]):
    d = gen()  # gen is a function reference — calling it like a callback
    ax.scatter(d[:, 0].numpy(), d[:, 1].numpy(), s=2, alpha=0.5)
    ax.set_title(name)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Train and Sample on All Datasets

Run this cell to train a fresh model on each dataset and compare real vs generated.

In [ ]:
def train_and_sample(data: torch.Tensor, name: str, n_steps: int = 5000):
    """Train a fresh model on `data` and show generated samples."""
    m = NoisePredictor()  # new model instance each time
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    bs = 256

    for step in range(n_steps):
        idx = torch.randint(0, len(data), (bs,))
        x_0 = data[idx]
        t = torch.randint(0, T, (bs,))
        noise = torch.randn_like(x_0)
        x_t, _ = forward_diffusion(x_0, t, noise)  # _ discards second return value
        loss = nn.functional.mse_loss(m(x_t, t), noise)
        opt.zero_grad()
        loss.backward()   # autograd computes gradients
        opt.step()        # optimizer updates weights

    # Sample from the trained model
    samples, _ = ddpm_sample(m, n_samples=2000)
    return samples


# List of tuples — like an array of [name, data] pairs in JS
datasets_to_try = [
    ("Moons", make_moons()),
    ("Circles", make_circles()),
    ("Figure-8", make_figure8()),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# enumerate() gives (index, value) — like .forEach((item, index) => ...) but index comes first
for col, (name, data) in enumerate(datasets_to_try):  # col = 0, 1, 2
    print(f"Training on {name}...")
    generated = train_and_sample(data, name)

    # axes[row, col] — 2D indexing into subplot grid
    # Real
    axes[0, col].scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=2, alpha=0.4)
    axes[0, col].set_title(f"{name} (Real)")
    axes[0, col].set_aspect("equal")
    axes[0, col].set_xlim(-2, 2)
    axes[0, col].set_ylim(-2, 2)

    # Generated
    axes[1, col].scatter(generated[:, 0].numpy(), generated[:, 1].numpy(), s=2, alpha=0.4, color="tab:orange")
    axes[1, col].set_title(f"{name} (Generated)")
    axes[1, col].set_aspect("equal")
    axes[1, col].set_xlim(-2, 2)
    axes[1, col].set_ylim(-2, 2)

plt.suptitle("Experiments: Real vs Generated", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

That's it! With just an MLP and a few lines of code, DDPM learns to reverse the noise process and generate samples from arbitrary 2D distributions.

For the full explanation of the math, variance schedules, and connections to score matching, check out the [mlearn diffusion lesson](https://mlearn.org).